# Example 12.8.1 / 1: Fit and Validate a Logistic Model

Companion notebook for **12.8.1 Optimization Models in ML Pipelines** in the AMPL lab Documentation, section "Fit and Validate a Logistic Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Fit and Validate a Logistic Model

Eight training observations record study hours and a pass label; four more are held out. The
feature is standardized with the training mean and standard deviation only. HiGHS does not solve `log` and `exp` objectives, so this example uses the free
Ipopt solver.
run.py sets `lambda`, solves, reads `intercept` and `beta` with `value()`, and computes the mean
log loss on both splits in Python. It finally makes the training labels perfectly separable and
solves without regularization.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set OBS;                  # training observations
set FEATURES;             # input features
param x {OBS, FEATURES};  # standardized feature value
param y {OBS} binary;     # class label, 0 or 1
param lambda >= 0;        # L2 regularization weight

var intercept;            # bias term
var beta {FEATURES};      # feature coefficients
var score {i in OBS} = intercept + sum {j in FEATURES} beta[j] * x[i,j];  # log-odds of class 1

minimize LogLoss:         # negative log-likelihood plus L2 penalty
    sum {i in OBS} (log(1 + exp(score[i])) - y[i] * score[i])
  + lambda * sum {j in FEATURES} beta[j]^2;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import math
hours = [1, 2, 3, 4, 5, 6, 7, 8]                # training feature
passed = [0, 0, 1, 0, 1, 1, 1, 1]               # training labels
mean = sum(hours) / len(hours)                  # training statistics only
std = math.sqrt(sum((h - mean) ** 2 for h in hours) / len(hours))

obs = [f"t{i}" for i in range(1, 9)]
ampl.set["OBS"] = obs
ampl.set["FEATURES"] = ["hours"]
ampl.param["x"] = {(o, "hours"): (h - mean) / std for o, h in zip(obs, hours)}
ampl.param["y"] = dict(zip(obs, passed))

In [5]:
# run.py
def mean_log_loss(b0, b1, xs, ys):
    total = 0
    for h, label in zip(xs, ys):
        p = 1 / (1 + math.exp(-(b0 + b1 * (h - mean) / std)))
        total -= label * math.log(p) + (1 - label) * math.log(1 - p)
    return total / len(xs)

valid_hours, valid_passed = [2.5, 4.5, 6.5, 7.5], [0, 1, 1, 1]
for lam in [0.01, 1, 10]:
    ampl.param["lambda"] = lam
    ampl.solve(solver="ipopt", verbose=False)
    b0, b1 = ampl.var["intercept"].value(), ampl.var["beta"]["hours"].value()
    print(lam, ampl.solve_result, round(b0, 4), round(b1, 4),
          round(mean_log_loss(b0, b1, hours, passed), 4),
          round(mean_log_loss(b0, b1, valid_hours, valid_passed), 4))

# Separable labels and no regularization.
ampl.param["lambda"] = 0
ampl.param["y"] = dict(zip(obs, [0, 0, 0, 0, 1, 1, 1, 1]))
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result, round(ampl.var["beta"]["hours"].value(), 2),
      ampl.obj["LogLoss"].value())

0.01 solved 1.2018 2.6952 0.3125 0.1437
1 solved 0.5812 0.7607 0.4569 0.3844
10 solved 0.5129 0.1297 0.6175 0.5548
solved 81.08 4.140557052778604e-08
